# Análise Global dos Resultados do Processo de Active Learning

Este notebook apresenta análises globais do desempenho do pipeline de Active Learning para segmentação de árvores.

**Análises incluídas:**
- Evolução da quantidade de instâncias e área segmentada durante as iterações
- Evolução das métricas de desempenho (Acurácia, F1-Score, KappaScore)
- Evolução temporal do conjunto de treino
- Distribuição de área por espécie
- F1-Score por espécie ao longo das iterações
- Visualizações de amostras de segmentação


## Setup e Imports


In [ ]:
import os
import yaml
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from glob import glob
import warnings
warnings.filterwarnings('ignore')

# Configurações de visualização
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_context("notebook", font_scale=1.2)
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['figure.dpi'] = 100

# Cores
COLORS = {
    'primary': '#2E86AB',
    'secondary': '#A23B72',
    'accent': '#F18F01',
    'success': '#06A77D',
    'warning': '#F77F00',
    'danger': '#D62828'
}

print("✓ Imports realizados com sucesso!")


## Configuração de Caminhos


In [ ]:
# Defina o caminho para a versão do experimento
VERSION = "v02"
DATA_PATH = Path(f"../../../bioflore_data/{VERSION}")

# Verificar se o caminho existe
if not DATA_PATH.exists():
    raise FileNotFoundError(f"Caminho não encontrado: {DATA_PATH}")

# Carregar args.yaml
args_path = DATA_PATH / "args.yaml"
with open(args_path, 'r') as f:
    args = yaml.safe_load(f)

print(f"✓ Versão: {VERSION}")
print(f"✓ Caminho dos dados: {DATA_PATH}")
print(f"✓ Número de iterações: {args['num_iter']}")


## Mapeamento de Classes para Espécies


In [ ]:
# Mapeamento de classes (1-4) para nomes de espécies
# Baseado na ordenação alfabética das espécies selecionadas
SPECIES_MAP = {
    1: 'Pterodon emarginatus',
    2: 'Qualea parviflora',
    3: 'Salvertia convallariodora',
    4: 'Tachigali aurea'
}

# Nomes curtos para visualizações
SPECIES_SHORT_NAMES = {
    1: 'P. emarginatus',
    2: 'Q. parviflora',
    3: 'S. convallariodora',
    4: 'T. aurea'
}

# Paleta de cores por espécie
SPECIES_COLORS = {
    1: '#E63946',  # Vermelho
    2: '#F77F00',  # Laranja
    3: '#06A77D',  # Verde
    4: '#2E86AB'   # Azul
}

print("✓ Mapeamento de espécies configurado:")
for class_id, species in SPECIES_MAP.items():
    print(f"  Classe {class_id}: {species}")


## Carregamento de Dados


In [ ]:
def load_global_metrics(data_path, num_iter):
    """
    Carrega métricas globais de todas as iterações.
    
    Returns:
        DataFrame com métricas globais por iteração
    """
    metrics_list = []
    
    for i in range(num_iter + 1):
        iter_folder = data_path / f"iter_{i:03d}"
        metrics_file = iter_folder / "global_metrics.yaml"
        
        if not metrics_file.exists():
            # Pular iterações sem métricas (ex: iter_000 pode não ter)
            continue
        
        with open(metrics_file, 'r') as f:
            metrics = yaml.safe_load(f)
        
        # Extrair métricas de treino e teste
        iter_data = {'iter': i}
        
        # Métricas de treino
        if 'global/train/Accuracy' in metrics:
            iter_data['train_accuracy'] = metrics['global/train/Accuracy']
            iter_data['train_kappa'] = metrics['global/train/KappaScore']
            iter_data['train_f1_avg'] = metrics['global/train/avgF1']
            iter_data['train_f1_weighted'] = metrics['global/train/avgF1_weighted']
            
            # F1 por classe
            for idx, f1 in enumerate(metrics['global/train/F1'], 1):
                iter_data[f'train_f1_class_{idx}'] = f1
        
        # Métricas de teste
        if 'global/test/Accuracy' in metrics:
            iter_data['test_accuracy'] = metrics['global/test/Accuracy']
            iter_data['test_kappa'] = metrics['global/test/KappaScore']
            iter_data['test_f1_avg'] = metrics['global/test/avgF1']
            iter_data['test_f1_weighted'] = metrics['global/test/avgF1_weighted']
            
            # F1 por classe
            for idx, f1 in enumerate(metrics['global/test/F1'], 1):
                iter_data[f'test_f1_class_{idx}'] = f1
        
        metrics_list.append(iter_data)
    
    df = pd.DataFrame(metrics_list)
    return df

# Carregar métricas
df_metrics = load_global_metrics(DATA_PATH, args['num_iter'])
print(f"✓ Métricas carregadas: {len(df_metrics)} iterações")
print(f"✓ Colunas disponíveis: {df_metrics.shape[1]}")
df_metrics.head()


In [ ]:
def load_instance_stats(data_path, num_iter):
    """
    Carrega estatísticas de instâncias (árvores) de todas as iterações.
    
    Returns:
        DataFrame com estatísticas de todas as instâncias
    """
    all_stats = []
    
    for i in range(num_iter + 1):
        iter_folder = data_path / f"iter_{i:03d}"
        stats_file = iter_folder / "all_regions_stats.parquet"
        
        if not stats_file.exists():
            # Pular iterações sem estatísticas (ex: iter_000 pode não ter)
            continue
        
        df = pd.read_parquet(stats_file)
        all_stats.append(df)
    
    df_all = pd.concat(all_stats, ignore_index=True)
    
    # Adicionar nomes de espécies
    df_all['species_name'] = df_all['tree_type'].map(SPECIES_MAP)
    df_all['species_short'] = df_all['tree_type'].map(SPECIES_SHORT_NAMES)
    
    return df_all

# Carregar estatísticas de instâncias
df_instances = load_instance_stats(DATA_PATH, args['num_iter'])
print(f"✓ Estatísticas de instâncias carregadas: {len(df_instances)} instâncias")
print(f"✓ Colunas: {list(df_instances.columns)}")
df_instances.head()


In [ ]:
def load_training_stats(data_path):
    """
    Carrega estatísticas de treinamento (loss e F1 por época).
    
    Returns:
        DataFrame com estatísticas de treinamento
    """
    stats_file = data_path / "training_stats.parquet"
    
    if not stats_file.exists():
        print(f"⚠ Arquivo não encontrado: {stats_file}")
        return None
    
    df = pd.read_parquet(stats_file)
    return df

# Carregar estatísticas de treinamento
df_training = load_training_stats(DATA_PATH)
if df_training is not None:
    print(f"✓ Estatísticas de treinamento carregadas: {len(df_training)} registros")
    print(f"✓ Colunas: {list(df_training.columns)}")
    df_training.head()
else:
    print("⚠ Estatísticas de treinamento não disponíveis")


## Big Numbers - Visão Geral


In [ ]:
# Estatísticas gerais da última iteração
last_iter = df_instances[df_instances['iter_num'] == args['num_iter']]

total_trees = len(last_iter)
total_area_m2 = last_iter['area'].sum()
total_area_ha = total_area_m2 / 10000

# Estatísticas por espécie
trees_by_species = last_iter.groupby('species_name').size()
area_by_species = last_iter.groupby('species_name')['area'].sum()

# Métricas de desempenho da última iteração
last_metrics = df_metrics[df_metrics['iter'] == args['num_iter']].iloc[0]

print("=" * 60)
print(f"RESUMO GERAL - ITERAÇÃO {args['num_iter']}")
print("=" * 60)
print(f"\n📊 TOTAL DE ÁRVORES IDENTIFICADAS: {total_trees}")
print(f"📐 ÁREA TOTAL SEGMENTADA: {total_area_m2:.2f} m² ({total_area_ha:.2f} ha)")
print(f"\n🎯 MÉTRICAS DE DESEMPENHO (Teste):")
print(f"   • Acurácia: {last_metrics['test_accuracy']:.2f}%")
print(f"   • F1-Score (média): {last_metrics['test_f1_avg']:.2f}%")
print(f"   • Kappa Score: {last_metrics['test_kappa']:.2f}%")
print(f"\n🌳 ÁRVORES POR ESPÉCIE:")
for species, count in trees_by_species.items():
    area_ha = area_by_species[species] / 10000
    print(f"   • {species}: {count} árvores ({area_ha:.2f} ha)")
print("=" * 60)


## Evolução da Quantidade de Instâncias e Área Segmentada


In [ ]:
# Calcular número de árvores por iteração
trees_per_iter = df_instances.groupby('iter_num').size().reset_index(name='n_trees')

# Calcular área total por iteração
area_per_iter = df_instances.groupby('iter_num')['area'].sum().reset_index(name='total_area_m2')
area_per_iter['total_area_ha'] = area_per_iter['total_area_m2'] / 10000

# Merge
evolution = trees_per_iter.merge(area_per_iter, on='iter_num')

# Visualização
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Gráfico 1: Número de árvores
ax1 = axes[0]
ax1.plot(evolution['iter_num'], evolution['n_trees'], 
         marker='o', linewidth=2.5, markersize=8, color=COLORS['primary'])
ax1.fill_between(evolution['iter_num'], evolution['n_trees'], alpha=0.3, color=COLORS['primary'])
ax1.set_xlabel('Iteração', fontsize=12, fontweight='bold')
ax1.set_ylabel('Número de Árvores', fontsize=12, fontweight='bold')
ax1.set_title('Evolução do Número de Árvores Identificadas', fontsize=14, fontweight='bold')
ax1.grid(True, alpha=0.3)
ax1.set_xlim(0, args['num_iter'])

# Adicionar valores no último ponto
last_point = evolution.iloc[-1]
ax1.annotate(f"{int(last_point['n_trees'])} árvores", 
             xy=(last_point['iter_num'], last_point['n_trees']),
             xytext=(10, 10), textcoords='offset points',
             fontsize=11, fontweight='bold',
             bbox=dict(boxstyle='round,pad=0.5', facecolor='yellow', alpha=0.7))

# Gráfico 2: Área total
ax2 = axes[1]
ax2.plot(evolution['iter_num'], evolution['total_area_ha'], 
         marker='s', linewidth=2.5, markersize=8, color=COLORS['success'])
ax2.fill_between(evolution['iter_num'], evolution['total_area_ha'], alpha=0.3, color=COLORS['success'])
ax2.set_xlabel('Iteração', fontsize=12, fontweight='bold')
ax2.set_ylabel('Área Total (ha)', fontsize=12, fontweight='bold')
ax2.set_title('Evolução da Área Total Segmentada', fontsize=14, fontweight='bold')
ax2.grid(True, alpha=0.3)
ax2.set_xlim(0, args['num_iter'])

# Adicionar valores no último ponto
ax2.annotate(f"{last_point['total_area_ha']:.2f} ha", 
             xy=(last_point['iter_num'], last_point['total_area_ha']),
             xytext=(10, 10), textcoords='offset points',
             fontsize=11, fontweight='bold',
             bbox=dict(boxstyle='round,pad=0.5', facecolor='yellow', alpha=0.7))

plt.tight_layout()
plt.show()

print(f"✓ Crescimento total: {evolution.iloc[0]['n_trees']} → {evolution.iloc[-1]['n_trees']} árvores")
print(f"✓ Incremento: {evolution.iloc[-1]['n_trees'] - evolution.iloc[0]['n_trees']} árvores ({((evolution.iloc[-1]['n_trees'] / evolution.iloc[0]['n_trees']) - 1) * 100:.1f}%)")


### Evolução por Espécie


In [ ]:
# Número de árvores por espécie e iteração
trees_by_species_iter = df_instances.groupby(['iter_num', 'species_short']).size().reset_index(name='n_trees')

# Visualização
fig, ax = plt.subplots(figsize=(14, 7))

for species_id, species_name in SPECIES_SHORT_NAMES.items():
    data = trees_by_species_iter[trees_by_species_iter['species_short'] == species_name]
    ax.plot(data['iter_num'], data['n_trees'], 
            marker='o', linewidth=2.5, markersize=7, 
            label=species_name, color=SPECIES_COLORS[species_id])

ax.set_xlabel('Iteração', fontsize=12, fontweight='bold')
ax.set_ylabel('Número de Árvores', fontsize=12, fontweight='bold')
ax.set_title('Evolução do Número de Árvores por Espécie', fontsize=14, fontweight='bold')
ax.legend(loc='upper left', fontsize=11, framealpha=0.9)
ax.grid(True, alpha=0.3)
ax.set_xlim(0, args['num_iter'])

plt.tight_layout()
plt.show()

# Tabela com evolução
pivot_table = trees_by_species_iter.pivot(index='species_short', columns='iter_num', values='n_trees')
print("\n📊 Tabela: Número de árvores por espécie e iteração")
print(pivot_table.fillna(0).astype(int))


## Evolução das Métricas de Desempenho


In [ ]:
# Visualização das principais métricas
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# 1. Acurácia
ax1 = axes[0, 0]
ax1.plot(df_metrics['iter'], df_metrics['train_accuracy'], 
         marker='o', linewidth=2.5, markersize=7, label='Treino', color=COLORS['primary'])
ax1.plot(df_metrics['iter'], df_metrics['test_accuracy'], 
         marker='s', linewidth=2.5, markersize=7, label='Teste', color=COLORS['danger'])
ax1.set_xlabel('Iteração', fontsize=11, fontweight='bold')
ax1.set_ylabel('Acurácia (%)', fontsize=11, fontweight='bold')
ax1.set_title('Evolução da Acurácia', fontsize=13, fontweight='bold')
ax1.legend(loc='lower right', fontsize=10)
ax1.grid(True, alpha=0.3)
ax1.set_xlim(0, args['num_iter'])
ax1.set_ylim(0, 100)

# 2. F1-Score médio
ax2 = axes[0, 1]
ax2.plot(df_metrics['iter'], df_metrics['train_f1_avg'], 
         marker='o', linewidth=2.5, markersize=7, label='Treino', color=COLORS['primary'])
ax2.plot(df_metrics['iter'], df_metrics['test_f1_avg'], 
         marker='s', linewidth=2.5, markersize=7, label='Teste', color=COLORS['danger'])
ax2.set_xlabel('Iteração', fontsize=11, fontweight='bold')
ax2.set_ylabel('F1-Score Médio (%)', fontsize=11, fontweight='bold')
ax2.set_title('Evolução do F1-Score Médio', fontsize=13, fontweight='bold')
ax2.legend(loc='lower right', fontsize=10)
ax2.grid(True, alpha=0.3)
ax2.set_xlim(0, args['num_iter'])
ax2.set_ylim(0, 100)

# 3. Kappa Score
ax3 = axes[1, 0]
ax3.plot(df_metrics['iter'], df_metrics['train_kappa'], 
         marker='o', linewidth=2.5, markersize=7, label='Treino', color=COLORS['primary'])
ax3.plot(df_metrics['iter'], df_metrics['test_kappa'], 
         marker='s', linewidth=2.5, markersize=7, label='Teste', color=COLORS['danger'])
ax3.set_xlabel('Iteração', fontsize=11, fontweight='bold')
ax3.set_ylabel('Kappa Score (%)', fontsize=11, fontweight='bold')
ax3.set_title('Evolução do Kappa Score', fontsize=13, fontweight='bold')
ax3.legend(loc='lower right', fontsize=10)
ax3.grid(True, alpha=0.3)
ax3.set_xlim(0, args['num_iter'])
ax3.set_ylim(0, 100)

# 4. F1-Score Weighted
ax4 = axes[1, 1]
ax4.plot(df_metrics['iter'], df_metrics['train_f1_weighted'], 
         marker='o', linewidth=2.5, markersize=7, label='Treino', color=COLORS['primary'])
ax4.plot(df_metrics['iter'], df_metrics['test_f1_weighted'], 
         marker='s', linewidth=2.5, markersize=7, label='Teste', color=COLORS['danger'])
ax4.set_xlabel('Iteração', fontsize=11, fontweight='bold')
ax4.set_ylabel('F1-Score Ponderado (%)', fontsize=11, fontweight='bold')
ax4.set_title('Evolução do F1-Score Ponderado', fontsize=13, fontweight='bold')
ax4.legend(loc='lower right', fontsize=10)
ax4.grid(True, alpha=0.3)
ax4.set_xlim(0, args['num_iter'])
ax4.set_ylim(0, 100)

plt.tight_layout()
plt.show()

# Resumo das métricas
print("\n📊 RESUMO DAS MÉTRICAS (Última Iteração):")
print(f"   Treino - Acurácia: {last_metrics['train_accuracy']:.2f}% | F1: {last_metrics['train_f1_avg']:.2f}% | Kappa: {last_metrics['train_kappa']:.2f}%")
print(f"   Teste  - Acurácia: {last_metrics['test_accuracy']:.2f}% | F1: {last_metrics['test_f1_avg']:.2f}% | Kappa: {last_metrics['test_kappa']:.2f}%")


### Comparação: F1-Score vs Número de Árvores


In [ ]:
# Gráfico com dois eixos Y
fig, ax1 = plt.subplots(figsize=(14, 7))

# Eixo 1: Número de árvores
color1 = COLORS['primary']
ax1.set_xlabel('Iteração', fontsize=12, fontweight='bold')
ax1.set_ylabel('Número de Árvores', fontsize=12, fontweight='bold', color=color1)
ax1.plot(evolution['iter_num'], evolution['n_trees'], 
         marker='o', linewidth=2.5, markersize=8, color=color1, label='Número de Árvores')
ax1.tick_params(axis='y', labelcolor=color1)
ax1.grid(True, alpha=0.3)
ax1.set_xlim(0, args['num_iter'])

# Eixo 2: F1-Score
ax2 = ax1.twinx()
color2 = COLORS['danger']
ax2.set_ylabel('F1-Score Médio (%) - Teste', fontsize=12, fontweight='bold', color=color2)
ax2.plot(df_metrics['iter'], df_metrics['test_f1_avg'], 
         marker='s', linewidth=2.5, markersize=8, color=color2, label='F1-Score (Teste)')
ax2.tick_params(axis='y', labelcolor=color2)

# Título
ax1.set_title('Relação entre Número de Árvores e F1-Score', fontsize=14, fontweight='bold')

# Legendas combinadas
lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, loc='upper left', fontsize=11, framealpha=0.9)

plt.tight_layout()
plt.show()

print(f"✓ Correlação entre número de árvores e F1-Score: {evolution['n_trees'].corr(df_metrics['test_f1_avg']):.3f}")


## Métricas por Componente

Nesta seção, analisamos as métricas de F1-Score calculadas a nível de **componente conectado**. 

Para cada componente conectado no ground truth (cada árvore rotulada), a classe predita é determinada pela **classe mais comum** dentro daquele componente. Isso permite avaliar se o modelo consegue identificar corretamente cada árvore individual como um todo, ao invés de apenas avaliar a precisão pixel a pixel.

### Comparação: F1-Score por Componente vs Número de Árvores



In [ ]:
# Carregar métricas de componente
def load_component_metrics(data_path, num_iter):
    """
    Carrega métricas de componentes de todas as iterações.
    
    Para cada componente conectado no ground truth, a classe predita é determinada
    pela classe mais comum dentro daquele componente.
    
    Returns:
        DataFrame com métricas de componentes por iteração
    """
    metrics_list = []
    
    for i in range(num_iter + 1):
        iter_folder = data_path / f"iter_{i:03d}"
        metrics_file = iter_folder / "global_component_metrics.yaml"
        
        if not metrics_file.exists():
            # Pular iterações sem métricas
            continue
        
        with open(metrics_file, 'r') as f:
            metrics = yaml.safe_load(f)
        
        # Extrair métricas de componente
        iter_data = {
            'iter': i,
            'component_accuracy': metrics.get('global/component/Accuracy', 0),
            'component_f1_avg': metrics.get('global/component/avgF1', 0),
            'component_avg_prec': metrics.get('global/component/avgPrec', 0),
            'component_avg_rec': metrics.get('global/component/avgRec', 0),
            'n_components': metrics.get('global/component/n_components', 0)
        }
        
        # F1 por classe (componente)
        if 'global/component/F1' in metrics:
            for idx, f1 in enumerate(metrics['global/component/F1'], 1):
                iter_data[f'component_f1_class_{idx}'] = f1
        
        metrics_list.append(iter_data)
    
    df = pd.DataFrame(metrics_list)
    return df

# Carregar métricas de componente
df_component_metrics = load_component_metrics(DATA_PATH, args['num_iter'])
print(f"✓ Métricas de componente carregadas: {len(df_component_metrics)} iterações")
print(f"✓ Colunas disponíveis: {df_component_metrics.shape[1]}")
df_component_metrics.head(10)


In [ ]:
# Gráfico com dois eixos Y: F1-Score por Componente vs Número de Árvores
fig, ax1 = plt.subplots(figsize=(14, 7))

# Eixo 1: Número de árvores
color1 = COLORS['primary']
ax1.set_xlabel('Iteração', fontsize=12, fontweight='bold')
ax1.set_ylabel('Número de Árvores', fontsize=12, fontweight='bold', color=color1)
ax1.plot(evolution['iter_num'], evolution['n_trees'], 
         marker='o', linewidth=2.5, markersize=8, color=color1, label='Número de Árvores')
ax1.tick_params(axis='y', labelcolor=color1)
ax1.grid(True, alpha=0.3)
ax1.set_xlim(0, args['num_iter'])

# Eixo 2: F1-Score por Componente
ax2 = ax1.twinx()
color2 = COLORS['accent']
ax2.set_ylabel('F1-Score Médio (%) - Por Componente', fontsize=12, fontweight='bold', color=color2)
ax2.plot(df_component_metrics['iter'], df_component_metrics['component_f1_avg'], 
         marker='D', linewidth=2.5, markersize=8, color=color2, label='F1-Score (Componente)')
ax2.tick_params(axis='y', labelcolor=color2)
ax2.set_ylim(0, 100)

# Título
ax1.set_title('Comparação: F1-Score por Componente vs Número de Árvores', fontsize=14, fontweight='bold')

# Legendas combinadas
lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, loc='upper left', fontsize=11, framealpha=0.9)

plt.tight_layout()
plt.show()

# Calcular correlação
merged_data = pd.merge(evolution, df_component_metrics, left_on='iter_num', right_on='iter', how='inner')
if len(merged_data) > 0:
    correlation = merged_data['n_trees'].corr(merged_data['component_f1_avg'])
    print(f"✓ Correlação entre número de árvores e F1-Score por componente: {correlation:.3f}")
else:
    print("⚠ Não foi possível calcular correlação - dados insuficientes")


In [ ]:
# Gráfico com dois eixos Y: Número de árvores vs F1-Score por Componente
fig, ax1 = plt.subplots(figsize=(14, 7))

# Eixo 1: Número de árvores
color1 = COLORS['primary']
ax1.set_xlabel('Iteração', fontsize=12, fontweight='bold')
ax1.set_ylabel('Número de Árvores', fontsize=12, fontweight='bold', color=color1)
ax1.plot(evolution['iter_num'], evolution['n_trees'], 
         marker='o', linewidth=2.5, markersize=8, color=color1, label='Número de Árvores')
ax1.tick_params(axis='y', labelcolor=color1)
ax1.grid(True, alpha=0.3)
ax1.set_xlim(0, args['num_iter'])

# Eixo 2: F1-Score por Componente
ax2 = ax1.twinx()
color2 = COLORS['accent']
ax2.set_ylabel('F1-Score Médio (%) - Por Componente', fontsize=12, fontweight='bold', color=color2)
ax2.plot(df_component_metrics['iter'], df_component_metrics['component_f1_avg'], 
         marker='D', linewidth=2.5, markersize=8, color=color2, label='F1-Score (Componente)')
ax2.tick_params(axis='y', labelcolor=color2)

# Título
ax1.set_title('Relação entre Número de Árvores e F1-Score por Componente', fontsize=14, fontweight='bold')

# Legendas combinadas
lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, loc='upper left', fontsize=11, framealpha=0.9)

plt.tight_layout()
plt.show()

# Calcular correlação
# Alinhar os dados por iteração
merged_data = pd.merge(evolution, df_component_metrics, left_on='iter_num', right_on='iter', how='inner')
if len(merged_data) > 0:
    correlation = merged_data['n_trees'].corr(merged_data['component_f1_avg'])
    print(f"✓ Correlação entre número de árvores e F1-Score por componente: {correlation:.3f}")
else:
    print("⚠ Não foi possível calcular correlação - dados insuficientes")


In [ ]:
# Comparação direta: F1-Score por Pixel vs F1-Score por Componente
fig, ax = plt.subplots(figsize=(14, 7))

# F1-Score por pixel (teste)
ax.plot(df_metrics['iter'], df_metrics['test_f1_avg'], 
        marker='o', linewidth=2.5, markersize=8, 
        label='F1-Score (Pixel)', color=COLORS['danger'])

# F1-Score por componente
ax.plot(df_component_metrics['iter'], df_component_metrics['component_f1_avg'], 
        marker='D', linewidth=2.5, markersize=8, 
        label='F1-Score (Componente)', color=COLORS['accent'])

ax.set_xlabel('Iteração', fontsize=12, fontweight='bold')
ax.set_ylabel('F1-Score Médio (%)', fontsize=12, fontweight='bold')
ax.set_title('Comparação: F1-Score por Pixel vs F1-Score por Componente', fontsize=14, fontweight='bold')
ax.legend(loc='lower right', fontsize=11, framealpha=0.9)
ax.grid(True, alpha=0.3)
ax.set_xlim(0, args['num_iter'])
ax.set_ylim(0, 100)

plt.tight_layout()
plt.show()

# Estatísticas comparativas
print("\n📊 COMPARAÇÃO DE F1-SCORE (Última Iteração):")
last_iter_num = df_metrics['iter'].max()
last_pixel_f1 = df_metrics[df_metrics['iter'] == last_iter_num]['test_f1_avg'].values[0]

if last_iter_num in df_component_metrics['iter'].values:
    last_component_f1 = df_component_metrics[df_component_metrics['iter'] == last_iter_num]['component_f1_avg'].values[0]
    print(f"   F1-Score (Pixel):      {last_pixel_f1:.2f}%")
    print(f"   F1-Score (Componente): {last_component_f1:.2f}%")
    print(f"   Diferença:             {last_pixel_f1 - last_component_f1:.2f} pontos percentuais")
else:
    print(f"   F1-Score (Pixel):      {last_pixel_f1:.2f}%")
    print(f"   F1-Score (Componente): Não disponível para iteração {last_iter_num}")


### F1-Score por Componente - Por Espécie


In [ ]:
# Preparar dados de F1-Score por componente por classe
f1_component_by_class = []

for iter_num in range(args['num_iter'] + 1):
    for class_id in range(1, 5):
        component_f1_col = f'component_f1_class_{class_id}'
        if component_f1_col in df_component_metrics.columns:
            f1_value = df_component_metrics[df_component_metrics['iter'] == iter_num][component_f1_col].values
            if len(f1_value) > 0:
                f1_component_by_class.append({
                    'iter': iter_num,
                    'class_id': class_id,
                    'species': SPECIES_SHORT_NAMES[class_id],
                    'f1_score_component': f1_value[0]
                })

df_f1_component_species = pd.DataFrame(f1_component_by_class)

# Visualização
fig, ax = plt.subplots(figsize=(14, 7))

for class_id, species_name in SPECIES_SHORT_NAMES.items():
    data = df_f1_component_species[df_f1_component_species['class_id'] == class_id]
    if len(data) > 0:
        ax.plot(data['iter'], data['f1_score_component'], 
                marker='D', linewidth=2.5, markersize=7, 
                label=species_name, color=SPECIES_COLORS[class_id])

ax.set_xlabel('Iteração', fontsize=12, fontweight='bold')
ax.set_ylabel('F1-Score por Componente (%)', fontsize=12, fontweight='bold')
ax.set_title('Evolução do F1-Score por Componente - Por Espécie', fontsize=14, fontweight='bold')
ax.legend(loc='lower right', fontsize=11, framealpha=0.9)
ax.grid(True, alpha=0.3)
ax.set_xlim(0, args['num_iter'])
ax.set_ylim(0, 105)

plt.tight_layout()
plt.show()

# Tabela de F1-Score por componente por espécie
if len(df_f1_component_species) > 0:
    print("\n📊 F1-SCORE POR COMPONENTE - POR ESPÉCIE:")
    pivot_f1_component = df_f1_component_species.pivot(index='species', columns='iter', values='f1_score_component')
    print(pivot_f1_component.round(2))
else:
    print("\n⚠ Dados de F1-Score por componente por espécie não disponíveis")


### Comparação: F1-Score Pixel vs Componente por Espécie


In [ ]:
# Comparação lado a lado: F1-Score por Pixel vs por Componente para cada espécie
fig, axes = plt.subplots(2, 2, figsize=(16, 12))
axes = axes.flatten()

for idx, (class_id, species_name) in enumerate(SPECIES_SHORT_NAMES.items()):
    ax = axes[idx]
    
    # Dados de F1-Score por pixel
    pixel_data = df_f1_species[df_f1_species['class_id'] == class_id]
    
    # Dados de F1-Score por componente
    component_data = df_f1_component_species[df_f1_component_species['class_id'] == class_id]
    
    # Plotar ambas as curvas
    if len(pixel_data) > 0:
        ax.plot(pixel_data['iter'], pixel_data['f1_score'], 
                marker='o', linewidth=2.5, markersize=7, 
                label='F1-Score (Pixel)', color=SPECIES_COLORS[class_id], alpha=0.7)
    
    if len(component_data) > 0:
        ax.plot(component_data['iter'], component_data['f1_score_component'], 
                marker='D', linewidth=2.5, markersize=7, 
                label='F1-Score (Componente)', color=SPECIES_COLORS[class_id], linestyle='--')
    
    ax.set_xlabel('Iteração', fontsize=11, fontweight='bold')
    ax.set_ylabel('F1-Score (%)', fontsize=11, fontweight='bold')
    ax.set_title(species_name, fontsize=12, fontweight='bold')
    ax.legend(loc='lower right', fontsize=10)
    ax.grid(True, alpha=0.3)
    ax.set_xlim(0, args['num_iter'])
    ax.set_ylim(0, 105)

plt.suptitle('Comparação: F1-Score por Pixel vs por Componente - Por Espécie', 
             fontsize=14, fontweight='bold', y=1.00)
plt.tight_layout()
plt.show()

# Estatísticas comparativas por espécie
print("\n📊 COMPARAÇÃO F1-SCORE (Última Iteração) - POR ESPÉCIE:")
last_iter_num = df_metrics['iter'].max()

for class_id, species_name in SPECIES_MAP.items():
    # F1 por pixel
    pixel_col = f'test_f1_class_{class_id}'
    if pixel_col in df_metrics.columns:
        pixel_f1 = df_metrics[df_metrics['iter'] == last_iter_num][pixel_col].values
        if len(pixel_f1) > 0:
            pixel_f1 = pixel_f1[0]
        else:
            pixel_f1 = None
    else:
        pixel_f1 = None
    
    # F1 por componente
    component_col = f'component_f1_class_{class_id}'
    if component_col in df_component_metrics.columns:
        component_f1 = df_component_metrics[df_component_metrics['iter'] == last_iter_num][component_col].values
        if len(component_f1) > 0:
            component_f1 = component_f1[0]
        else:
            component_f1 = None
    else:
        component_f1 = None
    
    print(f"\n{species_name}:")
    if pixel_f1 is not None:
        print(f"   F1-Score (Pixel):      {pixel_f1:.2f}%")
    else:
        print(f"   F1-Score (Pixel):      N/A")
    
    if component_f1 is not None:
        print(f"   F1-Score (Componente): {component_f1:.2f}%")
    else:
        print(f"   F1-Score (Componente): N/A")
    
    if pixel_f1 is not None and component_f1 is not None:
        print(f"   Diferença:             {pixel_f1 - component_f1:.2f} pontos percentuais")


## Distribuição de Área por Espécie


In [ ]:
# Boxplot da distribuição de área por espécie (última iteração)
fig, ax = plt.subplots(figsize=(14, 7))

# Preparar dados
plot_data = []
positions = []
colors_list = []

for idx, (species_id, species_name) in enumerate(SPECIES_SHORT_NAMES.items(), 1):
    species_data = last_iter[last_iter['tree_type'] == species_id]['area'].values
    if len(species_data) > 0:
        plot_data.append(species_data)
        positions.append(idx)
        colors_list.append(SPECIES_COLORS[species_id])

# Criar boxplot
bp = ax.boxplot(plot_data, positions=positions, widths=0.6, patch_artist=True,
                showmeans=True, meanline=True,
                boxprops=dict(linewidth=1.5),
                medianprops=dict(color='black', linewidth=2),
                meanprops=dict(color='red', linewidth=2, linestyle='--'),
                whiskerprops=dict(linewidth=1.5),
                capprops=dict(linewidth=1.5))

# Colorir boxes
for patch, color in zip(bp['boxes'], colors_list):
    patch.set_facecolor(color)
    patch.set_alpha(0.7)

# Configurações
ax.set_xlabel('Espécie', fontsize=12, fontweight='bold')
ax.set_ylabel('Área (m²)', fontsize=12, fontweight='bold')
ax.set_title(f'Distribuição de Área por Espécie (Iteração {args["num_iter"]})', 
             fontsize=14, fontweight='bold')
ax.set_xticks(positions)
ax.set_xticklabels([SPECIES_SHORT_NAMES[i+1] for i in range(len(positions))], rotation=15, ha='right')
ax.grid(True, alpha=0.3, axis='y')

# Legenda
from matplotlib.lines import Line2D
legend_elements = [
    Line2D([0], [0], color='black', linewidth=2, label='Mediana'),
    Line2D([0], [0], color='red', linewidth=2, linestyle='--', label='Média')
]
ax.legend(handles=legend_elements, loc='upper right', fontsize=10)

plt.tight_layout()
plt.show()

# Estatísticas descritivas
print("\n📊 ESTATÍSTICAS DE ÁREA POR ESPÉCIE (m²):")
for species_id, species_name in SPECIES_MAP.items():
    species_data = last_iter[last_iter['tree_type'] == species_id]['area']
    if len(species_data) > 0:
        print(f"\n{species_name}:")
        print(f"   Média: {species_data.mean():.2f} m² | Mediana: {species_data.median():.2f} m²")
        print(f"   Min: {species_data.min():.2f} m² | Max: {species_data.max():.2f} m²")
        print(f"   Desvio padrão: {species_data.std():.2f} m²")


### Evolução da Distribuição de Área


In [ ]:
# Comparar distribuição de área entre primeira e última iteração
first_iter_num = df_instances['iter_num'].min()  # Primeira iteração disponível
first_iter = df_instances[df_instances['iter_num'] == first_iter_num]

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Primeira iteração
ax1 = axes[0]
plot_data_first = []
positions = []
colors_list = []

for idx, (species_id, species_name) in enumerate(SPECIES_SHORT_NAMES.items(), 1):
    species_data = first_iter[first_iter['tree_type'] == species_id]['area'].values
    if len(species_data) > 0:
        plot_data_first.append(species_data)
        positions.append(idx)
        colors_list.append(SPECIES_COLORS[species_id])

bp1 = ax1.boxplot(plot_data_first, positions=positions, widths=0.6, patch_artist=True,
                  showmeans=True, meanline=True,
                  medianprops=dict(color='black', linewidth=2),
                  meanprops=dict(color='red', linewidth=2, linestyle='--'))

for patch, color in zip(bp1['boxes'], colors_list):
    patch.set_facecolor(color)
    patch.set_alpha(0.7)

ax1.set_xlabel('Espécie', fontsize=11, fontweight='bold')
ax1.set_ylabel('Área (m²)', fontsize=11, fontweight='bold')
ax1.set_title(f'Iteração {first_iter_num} (Inicial)', fontsize=13, fontweight='bold')
ax1.set_xticks(positions)
ax1.set_xticklabels([SPECIES_SHORT_NAMES[i+1] for i in range(len(positions))], rotation=15, ha='right')
ax1.grid(True, alpha=0.3, axis='y')

# Última iteração
ax2 = axes[1]
plot_data_last = []
positions = []
colors_list = []

for idx, (species_id, species_name) in enumerate(SPECIES_SHORT_NAMES.items(), 1):
    species_data = last_iter[last_iter['tree_type'] == species_id]['area'].values
    if len(species_data) > 0:
        plot_data_last.append(species_data)
        positions.append(idx)
        colors_list.append(SPECIES_COLORS[species_id])

bp2 = ax2.boxplot(plot_data_last, positions=positions, widths=0.6, patch_artist=True,
                  showmeans=True, meanline=True,
                  medianprops=dict(color='black', linewidth=2),
                  meanprops=dict(color='red', linewidth=2, linestyle='--'))

for patch, color in zip(bp2['boxes'], colors_list):
    patch.set_facecolor(color)
    patch.set_alpha(0.7)

ax2.set_xlabel('Espécie', fontsize=11, fontweight='bold')
ax2.set_ylabel('Área (m²)', fontsize=11, fontweight='bold')
ax2.set_title(f'Iteração {args["num_iter"]} (Final)', fontsize=13, fontweight='bold')
ax2.set_xticks(positions)
ax2.set_xticklabels([SPECIES_SHORT_NAMES[i+1] for i in range(len(positions))], rotation=15, ha='right')
ax2.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()


## F1-Score por Espécie


In [ ]:
# Preparar dados de F1-Score por classe
f1_by_class = []

for iter_num in range(args['num_iter'] + 1):
    for class_id in range(1, 5):
        test_f1_col = f'test_f1_class_{class_id}'
        if test_f1_col in df_metrics.columns:
            f1_value = df_metrics[df_metrics['iter'] == iter_num][test_f1_col].values
            if len(f1_value) > 0:
                f1_by_class.append({
                    'iter': iter_num,
                    'class_id': class_id,
                    'species': SPECIES_SHORT_NAMES[class_id],
                    'f1_score': f1_value[0]
                })

df_f1_species = pd.DataFrame(f1_by_class)

# Visualização
fig, ax = plt.subplots(figsize=(14, 7))

for class_id, species_name in SPECIES_SHORT_NAMES.items():
    data = df_f1_species[df_f1_species['class_id'] == class_id]
    ax.plot(data['iter'], data['f1_score'], 
            marker='o', linewidth=2.5, markersize=7, 
            label=species_name, color=SPECIES_COLORS[class_id])

ax.set_xlabel('Iteração', fontsize=12, fontweight='bold')
ax.set_ylabel('F1-Score (%)', fontsize=12, fontweight='bold')
ax.set_title('Evolução do F1-Score por Espécie (Teste)', fontsize=14, fontweight='bold')
ax.legend(loc='lower right', fontsize=11, framealpha=0.9)
ax.grid(True, alpha=0.3)
ax.set_xlim(0, args['num_iter'])
ax.set_ylim(0, 105)

plt.tight_layout()
plt.show()

# Tabela de F1-Score por espécie
print("\n📊 F1-SCORE POR ESPÉCIE (Teste):")
pivot_f1 = df_f1_species.pivot(index='species', columns='iter', values='f1_score')
print(pivot_f1.round(2))


### F1-Score e Número de Árvores por Espécie


In [ ]:
# Gráfico de barras agrupadas: F1-Score e número de árvores para iterações selecionadas
first_available_iter = df_instances['iter_num'].min()
selected_iters = [first_available_iter, 10, args['num_iter']]

# Preparar dados
data_for_plot = []
for iter_num in selected_iters:
    for class_id in range(1, 5):
        # F1-Score
        f1_data = df_f1_species[(df_f1_species['iter'] == iter_num) & 
                                 (df_f1_species['class_id'] == class_id)]
        
        # Número de árvores
        n_trees = len(df_instances[(df_instances['iter_num'] == iter_num) & 
                                    (df_instances['tree_type'] == class_id)])
        
        if len(f1_data) > 0:
            data_for_plot.append({
                'iter': f'Iter {iter_num}',
                'species': SPECIES_SHORT_NAMES[class_id],
                'class_id': class_id,
                'f1_score': f1_data['f1_score'].values[0],
                'n_trees': n_trees
            })

df_plot = pd.DataFrame(data_for_plot)

# Criar subplots
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Gráfico 1: F1-Score
ax1 = axes[0]
x = np.arange(len(selected_iters))
width = 0.2

for idx, (class_id, species_name) in enumerate(SPECIES_SHORT_NAMES.items()):
    species_data = df_plot[df_plot['class_id'] == class_id]
    f1_values = species_data['f1_score'].values
    ax1.bar(x + idx * width, f1_values, width, 
            label=species_name, color=SPECIES_COLORS[class_id], alpha=0.8)

ax1.set_xlabel('Iteração', fontsize=12, fontweight='bold')
ax1.set_ylabel('F1-Score (%)', fontsize=12, fontweight='bold')
ax1.set_title('F1-Score por Espécie', fontsize=13, fontweight='bold')
ax1.set_xticks(x + width * 1.5)
ax1.set_xticklabels([f'Iter {i}' for i in selected_iters])
ax1.legend(loc='lower right', fontsize=10)
ax1.grid(True, alpha=0.3, axis='y')
ax1.set_ylim(0, 105)

# Gráfico 2: Número de árvores
ax2 = axes[1]
for idx, (class_id, species_name) in enumerate(SPECIES_SHORT_NAMES.items()):
    species_data = df_plot[df_plot['class_id'] == class_id]
    n_trees_values = species_data['n_trees'].values
    ax2.bar(x + idx * width, n_trees_values, width, 
            label=species_name, color=SPECIES_COLORS[class_id], alpha=0.8)

ax2.set_xlabel('Iteração', fontsize=12, fontweight='bold')
ax2.set_ylabel('Número de Árvores', fontsize=12, fontweight='bold')
ax2.set_title('Número de Árvores por Espécie', fontsize=13, fontweight='bold')
ax2.set_xticks(x + width * 1.5)
ax2.set_xticklabels([f'Iter {i}' for i in selected_iters])
ax2.legend(loc='upper left', fontsize=10)
ax2.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

print(f"\n✓ Comparação entre iterações {selected_iters}")


## Evolução Temporal do Conjunto de Treino


In [ ]:
if df_training is not None:
    # Visualizar evolução do loss e F1 durante o treinamento
    fig, axes = plt.subplots(1, 2, figsize=(16, 6))
    
    # Loss por iteração
    ax1 = axes[0]
    for iter_num in df_training['iter'].unique():
        iter_data = df_training[df_training['iter'] == iter_num]
        ax1.plot(iter_data['epoch'], iter_data['train_loss'], 
                 linewidth=2, alpha=0.7, label=f'Iter {iter_num}')
    
    ax1.set_xlabel('Época', fontsize=12, fontweight='bold')
    ax1.set_ylabel('Loss', fontsize=12, fontweight='bold')
    ax1.set_title('Evolução do Loss Durante o Treinamento', fontsize=13, fontweight='bold')
    ax1.grid(True, alpha=0.3)
    ax1.legend(bbox_to_anchor=(1.05, 1), loc='upper left', fontsize=9, ncol=2)
    
    # F1 médio por iteração (última época)
    ax2 = axes[1]
    last_epoch_data = []
    for iter_num in df_training['iter'].unique():
        iter_data = df_training[df_training['iter'] == iter_num]
        last_epoch = iter_data[iter_data['epoch'] == iter_data['epoch'].max()]
        if len(last_epoch) > 0:
            f1_cols = [col for col in df_training.columns if col.startswith('f1_class_')]
            f1_mean = last_epoch[f1_cols].mean(axis=1).values[0]
            last_epoch_data.append({'iter': iter_num, 'f1_mean': f1_mean * 100})
    
    df_last_epoch = pd.DataFrame(last_epoch_data)
    ax2.plot(df_last_epoch['iter'], df_last_epoch['f1_mean'], 
             marker='o', linewidth=2.5, markersize=8, color=COLORS['success'])
    ax2.fill_between(df_last_epoch['iter'], df_last_epoch['f1_mean'], alpha=0.3, color=COLORS['success'])
    ax2.set_xlabel('Iteração', fontsize=12, fontweight='bold')
    ax2.set_ylabel('F1-Score Médio (%) - Última Época', fontsize=12, fontweight='bold')
    ax2.set_title('F1-Score Médio ao Final de Cada Treinamento', fontsize=13, fontweight='bold')
    ax2.grid(True, alpha=0.3)
    ax2.set_xlim(0, args['num_iter'])
    
    plt.tight_layout()
    plt.show()
    
    print(f"✓ Dados de treinamento visualizados para {len(df_training['iter'].unique())} iterações")
else:
    print("⚠ Dados de treinamento não disponíveis para visualização")


## Visualizações de Amostras de Segmentação


In [ ]:
from PIL import Image
import matplotlib.patches as mpatches

# Buscar imagens de visualização
viz_path = DATA_PATH / "visualization" / "synthetic_all_labels"

if viz_path.exists():
    # Listar imagens disponíveis
    image_files = sorted(list(viz_path.glob("*.png")))
    
    if len(image_files) > 0:
        print(f"✓ Encontradas {len(image_files)} imagens de visualização")
        
        # Selecionar algumas iterações para mostrar
        first_viz_iter = df_instances['iter_num'].min()
        selected_iters_viz = [first_viz_iter, 5, 10, 15, args['num_iter']]
        
        # Filtrar imagens das iterações selecionadas
        selected_images = []
        for iter_num in selected_iters_viz:
            # Procurar imagens da iteração
            iter_images = [f for f in image_files if f"_{iter_num:03d}_" in f.name]
            if len(iter_images) > 0:
                selected_images.append((iter_num, iter_images[0]))
        
        if len(selected_images) > 0:
            # Criar grid de visualizações
            n_images = len(selected_images)
            fig, axes = plt.subplots(1, n_images, figsize=(5*n_images, 5))
            
            if n_images == 1:
                axes = [axes]
            
            for idx, (iter_num, img_path) in enumerate(selected_images):
                img = Image.open(img_path)
                axes[idx].imshow(img)
                axes[idx].set_title(f'Iteração {iter_num}', fontsize=12, fontweight='bold')
                axes[idx].axis('off')
            
            plt.tight_layout()
            plt.show()
            
            print(f"✓ Visualizações exibidas para iterações: {[i for i, _ in selected_images]}")
        else:
            print("⚠ Nenhuma imagem encontrada para as iterações selecionadas")
    else:
        print("⚠ Nenhuma imagem de visualização encontrada")
else:
    print(f"⚠ Pasta de visualizações não encontrada: {viz_path}")


## Resumo Final


In [ ]:
print("=" * 80)
print("RELATÓRIO DE ANÁLISE - PIPELINE DE ACTIVE LEARNING")
print("=" * 80)
print(f"\n📁 Versão: {VERSION}")
print(f"📅 Número de iterações: {args['num_iter']}")
print(f"\n{'─' * 80}")
print("RESULTADOS FINAIS (Última Iteração)")
print('─' * 80)

print(f"\n🌳 ÁRVORES IDENTIFICADAS:")
print(f"   Total: {total_trees} árvores")
print(f"   Crescimento: {evolution.iloc[0]['n_trees']} → {total_trees} ({((total_trees / evolution.iloc[0]['n_trees']) - 1) * 100:.1f}%)")

print(f"\n📐 ÁREA SEGMENTADA:")
print(f"   Total: {total_area_m2:.2f} m² ({total_area_ha:.2f} ha)")

print(f"\n🎯 MÉTRICAS DE DESEMPENHO (Teste):")
print(f"   Acurácia: {last_metrics['test_accuracy']:.2f}%")
print(f"   F1-Score (média): {last_metrics['test_f1_avg']:.2f}%")
print(f"   F1-Score (ponderado): {last_metrics['test_f1_weighted']:.2f}%")
print(f"   Kappa Score: {last_metrics['test_kappa']:.2f}%")

print(f"\n📊 POR ESPÉCIE:")
for species_id, species_name in SPECIES_MAP.items():
    n_trees_species = len(last_iter[last_iter['tree_type'] == species_id])
    area_species = last_iter[last_iter['tree_type'] == species_id]['area'].sum() / 10000
    
    # F1-Score
    f1_col = f'test_f1_class_{species_id}'
    f1_value = last_metrics[f1_col] if f1_col in last_metrics else 0
    
    print(f"\n   {species_name}:")
    print(f"      • Árvores: {n_trees_species}")
    print(f"      • Área: {area_species:.2f} ha")
    print(f"      • F1-Score: {f1_value:.2f}%")

print(f"\n{'=' * 80}")
print("✓ Análise concluída com sucesso!")
print("=" * 80)
